

# OpenAI In-House Data Agent: End-to-End Technical Report

---

## Executive Technical Summary

OpenAI's in-house data agent is a **bespoke, internal-only AI-powered analytical system** architected to reason over OpenAI's entire data platform — spanning $600+$ petabytes across $70{,}000+$ datasets serving $3{,}500+$ internal users. The system is powered by **GPT-5.2**, leverages **Codex** for code-level enrichment, the **Embeddings API** for dense vector retrieval, and the **Evals API** for continuous quality assurance. It transforms natural-language questions into validated analytical insights through a multi-layered contextual reasoning pipeline, reducing time-to-insight from **days to minutes**.

This report dissects the agent's architecture, context engineering, self-learning mechanisms, evaluation framework, security model, and operational lessons with full technical depth.

---

## Table of Contents

1. [Motivation & Problem Formulation](#1-motivation--problem-formulation)
2. [System Architecture](#2-system-architecture)
3. [Multi-Layered Context Engineering](#3-multi-layered-context-engineering)
4. [Retrieval-Augmented Generation Pipeline](#4-retrieval-augmented-generation-pipeline)
5. [Agentic Reasoning & Closed-Loop Self-Correction](#5-agentic-reasoning--closed-loop-self-correction)
6. [Memory System & Continuous Learning](#6-memory-system--continuous-learning)
7. [Workflow Automation & Reusability](#7-workflow-automation--reusability)
8. [Evaluation Framework](#8-evaluation-framework)
9. [Security & Access Control Model](#9-security--access-control-model)
10. [Lessons Learned](#10-lessons-learned)
11. [Technical Analysis & Commentary](#11-technical-analysis--commentary)

---

## 1. Motivation & Problem Formulation

### 1.1 The Data Discovery Bottleneck

At OpenAI's operational scale, the data platform exhibits characteristics that make naive analytical approaches intractable:

| Dimension | Scale |
|---|---|
| Internal users | $3{,}500+$ |
| Total data volume | $600+$ PB |
| Distinct datasets | $70{,}000+$ |
| Functional teams served | Engineering, Data Science, Go-To-Market, Finance, Research |

The **core technical problem** is a compounding discovery-and-correctness bottleneck:

$$
T_{\text{insight}} = T_{\text{discovery}} + T_{\text{disambiguation}} + T_{\text{query\_construction}} + T_{\text{validation}} + T_{\text{interpretation}}
$$

where each component $T_i$ can independently dominate the total time $T_{\text{insight}}$.

### 1.2 Failure Modes in Manual Analysis

Even after correct table selection, analysts face **silent correctness failures** that are notoriously difficult to detect:

- **Many-to-many join explosions** — Uncontrolled fan-out producing inflated aggregates without obvious error signals.
- **Filter pushdown errors** — Predicates applied at incorrect points in the query plan, subtly altering result semantics.
- **Unhandled $\texttt{NULL}$ propagation** — SQL's three-valued logic ($\texttt{TRUE}$, $\texttt{FALSE}$, $\texttt{NULL}$) causing silent row exclusion in $\texttt{WHERE}$, $\texttt{JOIN}$, and aggregate operations.
- **Schema ambiguity** — Multiple tables with overlapping column names but different inclusion/exclusion criteria (e.g., logged-in vs. logged-out users, first-party vs. third-party traffic).

A representative failure case: a $180+$ line SQL statement joining customer geography data, deriving order-month fields, and computing monthly aggregates (order counts, gross revenue, revenue with tax, average ship-to-receipt days) — where **correct table selection and correct join semantics are non-obvious** even to experienced analysts.

### 1.3 Design Objective

The agent's design objective can be formalized as:

$$
\underset{\theta}{\operatorname{argmax}} \; \mathbb{E}_{q \sim \mathcal{Q}} \left[ \text{Correctness}(A_\theta(q)) \cdot \text{Speed}(A_\theta(q)) \cdot \text{Trust}(A_\theta(q)) \right]
$$

where $\mathcal{Q}$ is the distribution of natural-language analytical questions across all internal teams, $A_\theta$ is the agent parameterized by its context and reasoning pipeline, and the three factors represent **analytical correctness**, **time-to-insight**, and **user-verifiable transparency**.

---

## 2. System Architecture

### 2.1 High-Level Architecture

The system follows a **hub-and-spoke architecture** with a centralized Agent API orchestrating interactions between multiple entrypoints, context sources, and the reasoning backbone.

```
┌──────────────────────────────────────────────────────────────────────┐
│                         ENTRYPOINTS                                  │
│  ┌───────────┐  ┌────────────┐  ┌──────────────┐  ┌──────────────┐  │
│  │ Agent-UI  │  │ Slack Agent│  │ Local MCP    │  │ Remote MCP   │  │
│  │ (Web)     │  │            │  │ (IDE/Codex   │  │ (Internal    │  │
│  │           │  │            │  │  CLI)        │  │  ChatGPT)    │  │
│  └─────┬─────┘  └─────┬──────┘  └──────┬───────┘  └──────┬───────┘  │
│        │              │                │                  │          │
│        └──────────────┴────────┬───────┴──────────────────┘          │
│                                │                                     │
│                       ┌────────▼────────┐                            │
│                       │   Agent API     │                            │
│                       │  (Orchestrator) │                            │
│                       └──┬──────────┬───┘                            │
│              ┌───────────┘          └───────────┐                    │
│    ┌─────────▼──────────┐          ┌────────────▼──────────┐        │
│    │  Context Retrieval │          │     GPT-5.2 Model     │        │
│    │  ┌───────────────┐ │          │  (Reasoning Engine)   │        │
│    │  │ RAG Embeddings│ │          │                       │        │
│    │  │ Memory Store  │ │◄────────►│  ┌─────────────────┐  │        │
│    │  │ Data Warehouse│ │          │  │ Tool Invocation  │  │        │
│    │  │ Platform APIs │ │          │  │ Self-Correction  │  │        │
│    │  └───────────────┘ │          │  │ Code Generation  │  │        │
│    └────────────────────┘          │  └─────────────────┘  │        │
│                                    └───────────────────────┘        │
└──────────────────────────────────────────────────────────────────────┘
```

### 2.2 Core Technical Components

| Component | Technology | Role |
|---|---|---|
| **Reasoning Engine** | GPT-5.2 | Multi-step analytical reasoning, SQL generation, self-correction, natural-language synthesis |
| **Code Enrichment** | Codex | Codebase crawling to extract pipeline logic, table semantics, freshness guarantees, and business intent |
| **Vector Store** | OpenAI Embeddings API | Dense embedding generation for table metadata, annotations, and institutional knowledge |
| **Evaluation** | OpenAI Evals API | Continuous correctness assessment against golden SQL reference sets |
| **Protocol Layer** | MCP (Model Context Protocol) | Standardized tool/context connectivity across IDE, CLI, and ChatGPT surfaces |

### 2.3 Entrypoint Design

The agent is available across **four distinct interaction surfaces**, all routing through a unified Agent API:

1. **Agent-UI (Web)** — Full-featured browser interface with visualization, notebook publishing, and workflow management.
2. **Slack Agent** — Conversational interface embedded in existing team communication workflows.
3. **Local Agent-MCP (IDE/Codex CLI)** — Developer-oriented access via MCP, enabling data queries directly within coding environments.
4. **Remote Agent-MCP (Internal ChatGPT)** — MCP connector integration within OpenAI's internal ChatGPT deployment.

This multi-surface deployment strategy ensures the agent is available **where work already happens**, eliminating context-switching overhead.

### 2.4 End-to-End Query Flow

For any natural-language question $q$, the agent executes the following pipeline:

$$
q \xrightarrow{\text{parse}} \text{Intent} \xrightarrow{\text{retrieve}} \mathcal{C}(q) \xrightarrow{\text{reason}} \text{SQL}_{\text{gen}} \xrightarrow{\text{execute}} \mathcal{R} \xrightarrow{\text{validate}} \text{Insight}
$$

**Step-by-step:**

1. **Intent Parsing** — GPT-5.2 decomposes $q$ into analytical sub-goals (metrics, dimensions, filters, time ranges).
2. **Context Retrieval** — The RAG pipeline retrieves relevant table metadata, Codex enrichments, institutional knowledge, and applicable memories from the vector store.
3. **Reasoning & SQL Generation** — The model synthesizes retrieved context with its parametric knowledge to generate executable SQL, applying appropriate joins, filters, and aggregations.
4. **Execution** — Generated SQL is executed against the live data warehouse.
5. **Validation & Self-Correction** — Results are evaluated for plausibility (e.g., non-zero row counts, reasonable value ranges). If anomalies are detected, the agent loops back to step 2 or 3.
6. **Synthesis** — Final results are presented with explanatory context, assumptions, and direct links to underlying data for manual verification.

---

## 3. Multi-Layered Context Engineering

The agent's analytical quality is fundamentally determined by the **richness and accuracy of its grounding context**. The system implements a **six-layer context hierarchy**, each layer addressing a distinct category of information asymmetry.

### 3.1 Layer Architecture Overview

```
┌─────────────────────────────────────────────────────┐
│            Layer 6: Runtime Context                  │  ← Live
│  ┌─────────────────────────────────────────────┐    │
│  │         Layer 5: Memory                     │    │  ← Persistent
│  │  ┌─────────────────────────────────────┐    │    │
│  │  │    Layer 4: Institutional Knowledge │    │    │  ← Organizational
│  │  │  ┌─────────────────────────────┐    │    │    │
│  │  │  │  Layer 3: Codex Enrichment  │    │    │    │  ← Code-derived
│  │  │  │  ┌─────────────────────┐    │    │    │    │
│  │  │  │  │ Layer 2: Human      │    │    │    │    │  ← Expert-curated
│  │  │  │  │ Annotations         │    │    │    │    │
│  │  │  │  │ ┌───────────────┐   │    │    │    │    │
│  │  │  │  │ │ Layer 1:      │   │    │    │    │    │  ← Schema-derived
│  │  │  │  │ │ Table Usage   │   │    │    │    │    │
│  │  │  │  │ └───────────────┘   │    │    │    │    │
│  │  │  │  └─────────────────────┘    │    │    │    │
│  │  │  └─────────────────────────────┘    │    │    │
│  │  └─────────────────────────────────────┘    │    │
│  └─────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────┘
```

### 3.2 Layer 1: Table Usage (Schema & Query-Derived Metadata)

**Information Source:** Data warehouse catalog, query logs, lineage graphs

**What it captures:**

- **Schema metadata** — Column names, data types, primary keys, partition schemes, and table-level statistics.
- **Table lineage** — Directed acyclic graph (DAG) of upstream sources and downstream consumers, capturing how tables relate through ETL/ELT pipelines.
- **Historical query patterns** — Ingested production queries revealing canonical join patterns, commonly applied filters, and frequently co-accessed table combinations.

**Technical rationale:** Schema metadata provides the structural skeleton for SQL generation. Lineage graphs enable the agent to understand **table provenance** — critical for distinguishing between raw event tables, intermediate aggregates, and final reporting views. Historical query patterns serve as a **prior distribution** over likely query structures:

$$
P(\text{SQL}_{\text{correct}} \mid q) \propto P(\text{SQL}_{\text{correct}} \mid q, \mathcal{S}, \mathcal{L}, \mathcal{H})
$$

where $\mathcal{S}$ is schema metadata, $\mathcal{L}$ is lineage, and $\mathcal{H}$ is historical query patterns.

### 3.3 Layer 2: Human Annotations (Expert-Curated Semantics)

**Information Source:** Domain experts across data engineering, analytics, and product teams

**What it captures:**

- Curated natural-language descriptions of **table purpose**, **column semantics**, and **business meaning**.
- Known **caveats, edge cases, and data quality issues** (e.g., "this table excludes logged-out users," "revenue column is pre-tax for region X but post-tax for region Y").
- **Intent documentation** — why a table was created, not just what it contains.

**Technical rationale:** Metadata alone is structurally insufficient for semantic disambiguation. Two tables with identical schemas can have radically different semantics based on their creation context. Human annotations bridge the **semantic gap** between structural metadata and business meaning:

$$
\text{Semantic\_Clarity}(t) = f(\mathcal{S}(t), \mathcal{A}(t)) \gg f(\mathcal{S}(t))
$$

where $\mathcal{A}(t)$ represents human annotations for table $t$.

### 3.4 Layer 3: Codex Enrichment (Code-Derived Deep Semantics)

**Information Source:** OpenAI's production codebase, crawled and analyzed by Codex

**What it captures:**

This is the **most architecturally novel** context layer. By analyzing the actual pipeline code that produces each table, the agent derives:

| Enrichment Dimension | Description |
|---|---|
| **Table purpose** | Natural-language summary derived from pipeline logic |
| **Grain & primary keys** | The atomic unit of each row (e.g., per-user-per-day, per-session) |
| **Downstream usage patterns** | How other systems consume the table (Spark jobs, Python scripts, dashboards) |
| **Alternative table options** | Related tables that could serve similar analytical needs |
| **Data freshness** | Update frequency, SLA guarantees, and staleness thresholds |
| **Value uniqueness** | Cardinality characteristics of key columns |
| **Scope & exclusions** | What the table explicitly includes and excludes (e.g., first-party ChatGPT traffic only) |

**Pipeline Architecture:**

```
Popular/Critical Tables
        │
        ▼
┌───────────────────────────────────┐
│     Codex Task Distribution       │
│  ┌───────┐ ┌───────┐ ┌────────┐  │
│  │Purpose│ │Grain &│ │Downstream│ │
│  │Extract│ │Keys   │ │Usage    │  │
│  └───┬───┘ └───┬───┘ └────┬───┘  │
│  ┌───▼───┐ ┌───▼───┐ ┌────▼───┐  │
│  │Alt.   │ │Fresh- │ │Scope & │  │
│  │Tables │ │ness   │ │Excludes│  │
│  └───┬───┘ └───┬───┘ └────┬───┘  │
│      └─────────┴──────────┘      │
│              │                    │
│              ▼                    │
│     Unified Enrichment Record     │
└───────────────────────────────────┘
```

**Technical rationale:** This layer operationalizes a critical insight captured in Lesson #3 ("Meaning Lives in Code"): **the true semantics of a dataset are defined by the transformation logic that produces it**, not by its schema or metadata. Pipeline code encodes:

- **Filtering assumptions** (which records are included/excluded)
- **Temporal semantics** (event time vs. processing time, backfill behavior)
- **Business logic** (how metrics are computed, what edge cases are handled)

This information is **impossible to infer from schemas, column names, or even historical queries alone**.

**Automatic refresh:** Codex enrichment is regenerated automatically, ensuring context remains current without manual maintenance overhead — a critical property given the velocity of schema and pipeline changes at OpenAI's scale.

### 3.5 Layer 4: Institutional Knowledge (Organizational Context)

**Information Source:** Slack, Google Docs, Notion

**What it captures:**

- **Product launches** and their data implications (new events, schema changes, metric definition updates)
- **Reliability incidents** and their impact on data quality (logging outages, undercounted metrics)
- **Internal codenames and tool mappings** (translating colloquial references to formal system names)
- **Canonical metric definitions** (official computation logic for KPIs like DAU, WAU, revenue)

**Technical implementation:**

1. Documents are **ingested** from source systems via API connectors.
2. Content is **chunked, embedded, and stored** with associated metadata and permission labels.
3. A **retrieval service** handles access control, caching, and relevance ranking at query time.

**Illustrative example:** When a user asks about a usage dip in December, the agent retrieves institutional knowledge about a **logging issue starting November 13, 2025** — a fact that exists only in incident reports and Slack threads, not in any table metadata. Without this context, the agent would interpret the dip as a genuine usage decline rather than a data quality artifact.

### 3.6 Layer 5: Memory (Persistent Learned Corrections)

**Information Source:** User interactions, corrections, and discovered nuances

**What it captures:**

- **Non-obvious corrections** — Filters, constraints, and business rules that are critical for correctness but difficult to infer from other layers.
- **Disambiguation resolutions** — How ambiguous terms or requests should be interpreted in specific contexts.
- **Query patterns** — Learned optimal approaches for recurring analytical patterns.

**Memory taxonomy:**

| Scope | Visibility | Use Case |
|---|---|---|
| **Global** | All users | Organization-wide corrections (e.g., canonical metric filters) |
| **Personal** | Individual user | User-specific preferences and frequently asked patterns |

**Technical rationale:** Memory addresses the **long-tail of contextual knowledge** — corrections and nuances that are:

1. **Too specific** for human annotations (e.g., a particular experiment gate requires matching against a specific string)
2. **Too dynamic** for Codex enrichment (corrections discovered during interactive sessions)
3. **Too tacit** for institutional documentation (knowledge that exists only in individual analysts' heads)

**Interaction model:**

- When the agent receives a correction or discovers a nuance, it **proactively prompts** the user to save the learning.
- Users can also **manually create and edit** memories.
- Memories are surfaced at query time through the same RAG retrieval pipeline.

The memory system ensures that future queries begin from a **monotonically improving baseline**:

$$
\text{Accuracy}(A_\theta, t+1) \geq \text{Accuracy}(A_\theta, t)
$$

as corrections accumulate over time (assuming memory retrieval relevance is maintained).

### 3.7 Layer 6: Runtime Context (Live Data Inspection)

**Information Source:** Live data warehouse queries, metadata service, Airflow, Spark

**What it captures:**

- **Real-time schema validation** — Confirming that expected columns and types are present.
- **Live data sampling** — Inspecting actual values to verify assumptions about data content.
- **Pipeline status** — Current run status, last successful update timestamp, and pending backfills.
- **Cross-system metadata** — Information from orchestration (Airflow), compute (Spark), and metadata services that exists outside the warehouse.

**Technical rationale:** Runtime context serves as the **fallback and validation layer**. When:

- No prior context exists for a table (new or rarely-used datasets)
- Existing information may be stale (recently modified tables)
- Generated SQL produces unexpected results (requiring live data inspection for debugging)

the agent issues live queries to verify its understanding against ground truth.

---

## 4. Retrieval-Augmented Generation Pipeline

### 4.1 Offline Preprocessing

A **daily batch pipeline** aggregates context from Layers 1–5 into a unified, normalized representation:

```
┌────────────────────┐
│  Table Usage       │──┐
│  (Layer 1)         │  │
├────────────────────┤  │
│  Human Annotations │  │    ┌─────────────────────┐    ┌──────────────┐
│  (Layer 2)         │──┼───►│  Normalization &     │───►│  OpenAI      │
├────────────────────┤  │    │  Aggregation         │    │  Embeddings  │
│  Codex Enrichment  │  │    │  Pipeline            │    │  API         │
│  (Layer 3)         │──┤    └─────────────────────┘    └──────┬───────┘
├────────────────────┤  │                                      │
│  Institutional     │  │                                      ▼
│  Knowledge (L4)    │──┤                              ┌───────────────┐
├────────────────────┤  │                              │  Vector Store  │
│  Memory            │──┘                              │  (Embeddings)  │
│  (Layer 5)         │                                 └───────────────┘
└────────────────────┘
```

**Processing steps:**

1. **Extraction** — Raw metadata, annotations, enrichments, documents, and memories are pulled from source systems.
2. **Normalization** — Heterogeneous formats are unified into a canonical schema with consistent field naming, typing, and structure.
3. **Chunking** — Long documents and enrichment records are split into retrieval-optimized chunks with appropriate overlap.
4. **Embedding** — Each chunk is converted to a dense vector representation $\mathbf{e} \in \mathbb{R}^d$ using the OpenAI Embeddings API.
5. **Indexing** — Embeddings are stored in a vector index optimized for approximate nearest neighbor (ANN) retrieval.

### 4.2 Online Retrieval

At query time, given a natural-language question $q$:

1. **Query embedding:** $\mathbf{e}_q = \text{Embed}(q) \in \mathbb{R}^d$
2. **Semantic search:** Retrieve top-$k$ context chunks by cosine similarity:

$$
\mathcal{C}_k(q) = \underset{|\mathcal{S}| = k}{\operatorname{argmax}} \sum_{c \in \mathcal{S}} \frac{\mathbf{e}_q \cdot \mathbf{e}_c}{\|\mathbf{e}_q\| \cdot \|\mathbf{e}_c\|}
$$

3. **Context assembly:** Retrieved chunks are concatenated with the original query and any active memories to form the full prompt context for GPT-5.2.
4. **Augmented generation:** GPT-5.2 reasons over the assembled context to generate SQL, explanations, and analytical insights.

**Design properties:**

- **Scalability** — RAG avoids loading all $70{,}000+$ table metadata into the context window. Only the most relevant $k$ chunks are retrieved, keeping latency **predictable and low** regardless of total dataset count.
- **Freshness** — Daily pipeline re-embedding ensures context reflects recent schema changes and new annotations within $\leq 24$ hours.
- **Precision** — Dense retrieval over enriched representations (not raw metadata) yields higher relevance than keyword-based search.

---

## 5. Agentic Reasoning & Closed-Loop Self-Correction

### 5.1 Reasoning Architecture

The agent does **not** follow a fixed, linear script. Instead, it implements an **agentic reasoning loop** powered by GPT-5.2's chain-of-thought capabilities:

```
                    ┌──────────────────────────┐
                    │     User Question q      │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │   Decompose into         │
                    │   Sub-Goals              │
                    └────────────┬─────────────┘
                                 │
                                 ▼
               ┌─────────────────────────────────┐
               │   For each sub-goal:            │
               │   1. Retrieve relevant context  │
               │   2. Generate SQL / analysis    │
               │   3. Execute & inspect results  │◄──────┐
               │   4. Evaluate plausibility      │       │
               └───────────────┬─────────────────┘       │
                               │                         │
                          ┌────▼────┐                     │
                          │ Results │                     │
                          │ Valid?  │                     │
                          └────┬────┘                     │
                       Yes │      │ No                    │
                           │      └───────────────────────┘
                           ▼           (Diagnose, adjust,
                    ┌──────────────┐    retry)
                    │  Synthesize  │
                    │  Final       │
                    │  Insight     │
                    └──────────────┘
```

### 5.2 Self-Correction Mechanism

The closed-loop self-correction is a **critical differentiator** from static text-to-SQL systems. When the agent detects anomalous intermediate results:

| Signal | Diagnosis | Correction Action |
|---|---|---|
| Zero rows returned | Incorrect join condition or overly restrictive filter | Relax filters, verify join keys, inspect actual column values |
| Implausible magnitudes | Wrong aggregation level or missing deduplication | Check grain, add $\texttt{DISTINCT}$, verify group-by columns |
| Unexpected $\texttt{NULL}$ prevalence | Outer join producing nulls or data quality issue | Switch join type, add $\texttt{COALESCE}$, flag data quality |
| Schema mismatch | Referenced column doesn't exist in target table | Re-retrieve context, select alternative table |
| Timeout / resource exhaustion | Inefficient query plan | Add partition pruning, restructure CTEs, limit scan scope |

This process is formally analogous to a **control loop** with error detection and corrective feedback:

$$
\text{SQL}_{i+1} = \text{SQL}_i + \Delta(\text{Diagnose}(\mathcal{R}_i, \text{Expected}))
$$

where $\mathcal{R}_i$ is the result of executing $\text{SQL}_i$ and $\Delta$ is the corrective adjustment derived from diagnostic reasoning.

### 5.3 Multi-Turn Conversational Reasoning

The agent maintains **full conversational context** across turns, supporting:

- **Follow-up refinement** — "Now break this down by region" without restating the original query.
- **Mid-analysis redirection** — Users can interrupt and redirect the agent's analytical path.
- **Clarification dialogues** — When instructions are ambiguous, the agent asks targeted clarifying questions.
- **Sensible defaults** — When no clarification is provided, the agent applies reasonable assumptions (e.g., "last 30 days" for unspecified date ranges) and **explicitly states** the assumption.

This interaction model mirrors the dynamics of working with a skilled human analyst — **collaborative, iterative, and context-aware** — rather than a stateless query engine.

---

## 6. Memory System & Continuous Learning

### 6.1 Memory Architecture

The memory system implements a **persistent, retrieval-augmented learning loop** that allows the agent to improve monotonically over time:

```
┌──────────────────────────────────────────────────────┐
│                   Memory Lifecycle                    │
│                                                      │
│  Discovery            Storage            Retrieval   │
│  ┌────────────┐     ┌────────────┐     ┌──────────┐ │
│  │ User       │     │ Global     │     │ RAG      │ │
│  │ correction │────►│ Memory     │────►│ Retrieval│ │
│  │            │     │ Store      │     │ at Query │ │
│  │ Agent      │     ├────────────┤     │ Time     │ │
│  │ discovery  │────►│ Personal   │     │          │ │
│  │            │     │ Memory     │     │          │ │
│  │ Manual     │     │ Store      │     │          │ │
│  │ creation   │────►│            │     │          │ │
│  └────────────┘     └────────────┘     └──────────┘ │
└──────────────────────────────────────────────────────┘
```

### 6.2 Memory Content Categories

The memory system specifically targets knowledge that **falls through the gaps** of other context layers:

1. **Filter corrections** — "To query experiment X, filter on $\texttt{gate\_name = 'exp\_x\_v3'}$, not a fuzzy string match."
2. **Metric definitions** — "ChatGPT WAU is computed as $\texttt{COUNT(DISTINCT user\_id)}$ over a 7-day rolling window, excluding bot traffic."
3. **Table preferences** — "For ChatGPT top-level metrics, use $\texttt{metrics.chatgpt\_daily}$, not $\texttt{events.chatgpt\_raw}$."
4. **Known data issues** — "Table X is missing data between 2025-11-13 and 2025-12-01 due to logging incident #4521."

### 6.3 Memory Governance

| Property | Implementation |
|---|---|
| **Scope control** | Global memories affect all users; personal memories are user-scoped |
| **Editability** | Users can create, edit, and delete memories through the agent interface |
| **Transparency** | Memory saves are prompted with explicit content preview before confirmation |
| **Conflict resolution** | More recent memories take precedence; contradictions are flagged for human review |

---

## 7. Workflow Automation & Reusability

### 7.1 Workflow Abstraction

Observing that users frequently executed **identical or structurally similar analyses** (e.g., weekly business reports, table validation checks), the agent introduced **workflows** — reusable instruction sets that encode:

- **Analytical template** — The sequence of queries, transformations, and visualizations.
- **Context bindings** — Which tables, metrics, and filters to use.
- **Best practices** — Validated approaches for common analytical patterns.

### 7.2 Workflow Benefits

$$
T_{\text{repeat}} = \frac{T_{\text{initial}}}{\alpha}, \quad \alpha \gg 1
$$

where $T_{\text{initial}}$ is the time for the first execution and $\alpha$ is the acceleration factor from workflow reuse.

Additional benefits:

- **Consistency** — All users running the same workflow get identical methodology and results.
- **Knowledge encoding** — Expert analytical approaches are captured once and reused across the organization.
- **Onboarding acceleration** — New team members can execute complex analyses immediately using pre-built workflows.

---

## 8. Evaluation Framework

### 8.1 Evaluation Architecture

The evaluation system is built on the **OpenAI Evals API** and implements continuous quality monitoring:

```
┌───────────────────────────────────────────────────────────────┐
│                  Evaluation Pipeline                          │
│                                                               │
│  ┌─────────────────┐    ┌─────────────────┐                   │
│  │  Curated Q&A    │    │  Agent Query    │                   │
│  │  Eval Pairs     │    │  Generation    │                   │
│  │  ┌───────────┐  │    │  Endpoint      │                   │
│  │  │ Question  │──┼───►│  ┌───────────┐ │                   │
│  │  │ Golden SQL│  │    │  │ Generated │ │                   │
│  │  │ Expected  │  │    │  │ SQL       │ │                   │
│  │  │ Result    │  │    │  │ Actual    │ │                   │
│  │  └───────────┘  │    │  │ Result    │ │                   │
│  └────────┬────────┘    │  └─────┬─────┘ │                   │
│           │             └────────┼────────┘                   │
│           │                      │                            │
│           ▼                      ▼                            │
│  ┌──────────────────────────────────────────┐                │
│  │         OpenAI Evals Grader              │                │
│  │  ┌────────────────┐ ┌─────────────────┐  │                │
│  │  │ DataFrame      │ │ SQL Semantic    │  │                │
│  │  │ Comparison     │ │ Comparison      │  │                │
│  │  └────────┬───────┘ └────────┬────────┘  │                │
│  │           └──────────┬───────┘           │                │
│  │                      ▼                   │                │
│  │              ┌──────────────┐            │                │
│  │              │ Final Score  │            │                │
│  │              │ + Reasoning  │            │                │
│  │              └──────────────┘            │                │
│  └──────────────────────────────────────────┘                │
└───────────────────────────────────────────────────────────────┘
```

### 8.2 Evaluation Methodology

Each evaluation instance consists of a **question-answer-SQL triple** $(q_i, r_i^*, \text{SQL}_i^*)$ where:

- $q_i$ — Natural-language analytical question
- $\text{SQL}_i^*$ — Manually authored "golden" SQL producing the expected result
- $r_i^* = \text{Execute}(\text{SQL}_i^*)$ — The expected result set

**Evaluation process:**

1. Submit $q_i$ to the agent's query-generation endpoint → receive $\text{SQL}_i^{\text{gen}}$
2. Execute $\text{SQL}_i^{\text{gen}}$ → obtain $r_i^{\text{gen}}$
3. Compare $(r_i^{\text{gen}}, \text{SQL}_i^{\text{gen}})$ against $(r_i^*, \text{SQL}_i^*)$

### 8.3 Comparison Strategy

**Why not naive string matching:** Syntactically different SQL can be **semantically equivalent**. For example:

```sql
-- Golden SQL
SELECT COUNT(DISTINCT user_id) AS dau
FROM events
WHERE event_date = '2025-01-15'
  AND platform = 'chatgpt';

-- Generated SQL (semantically equivalent)
WITH filtered AS (
    SELECT user_id
    FROM events
    WHERE platform = 'chatgpt'
      AND event_date = '2025-01-15'
)
SELECT COUNT(DISTINCT user_id) AS dau
FROM filtered;
```

The Evals grader accounts for:

| Comparison Dimension | Handling |
|---|---|
| **SQL syntax variation** | Structural differences that preserve semantics are treated as correct |
| **Column ordering** | Extra or reordered columns that don't affect the core answer are tolerated |
| **Numerical precision** | Floating-point differences within $\epsilon$-tolerance are accepted |
| **Row ordering** | Unless ordering is explicitly part of the question, row order is ignored |

The grader produces a **score** and an **explanation**, capturing both correctness and the nature of any acceptable variation.

### 8.4 Evaluation Deployment Model

Evals serve dual purposes:

| Mode | Purpose | Trigger |
|---|---|---|
| **Development** | Regression detection during agent iteration | Every code change / prompt update |
| **Production** | Canary monitoring for quality drift | Continuous scheduled execution |

This mirrors the software engineering practice of **continuous integration / continuous deployment (CI/CD)** applied to AI agent quality:

$$
\text{Deploy}(\theta_{\text{new}}) \iff \text{Score}(\theta_{\text{new}}) \geq \text{Score}(\theta_{\text{current}}) - \delta
$$

where $\delta$ is the maximum acceptable regression threshold.

---

## 9. Security & Access Control Model

### 9.1 Design Principle: Pure Pass-Through

The agent implements a **zero-privilege-escalation** security model:

```
┌──────────────────────────────────────────────────┐
│              Security Architecture                │
│                                                   │
│  User Request                                    │
│       │                                          │
│       ▼                                          │
│  ┌──────────────────────┐                        │
│  │  Authentication      │ ← Existing identity    │
│  │  (SSO / Internal)    │   system               │
│  └──────────┬───────────┘                        │
│             │                                    │
│             ▼                                    │
│  ┌──────────────────────┐                        │
│  │  Authorization       │ ← Same permissions     │
│  │  (Pass-Through)      │   as direct data       │
│  │                      │   warehouse access     │
│  └──────────┬───────────┘                        │
│             │                                    │
│        ┌────▼────┐                               │
│        │ Access  │                               │
│        │ Granted?│                               │
│        └────┬────┘                               │
│       Yes │     │ No                             │
│           │     │                                │
│           ▼     ▼                                │
│    ┌──────────┐ ┌────────────────────┐           │
│    │ Execute  │ │ Flag missing       │           │
│    │ Query    │ │ access OR          │           │
│    │          │ │ fall back to       │           │
│    │          │ │ authorized         │           │
│    │          │ │ alternative tables │           │
│    └──────────┘ └────────────────────┘           │
└──────────────────────────────────────────────────┘
```

**Key properties:**

- **No privilege escalation** — The agent cannot access any data that the requesting user cannot access directly.
- **Graceful degradation** — When access is denied, the agent either flags the restriction or identifies alternative authorized datasets that can answer the question.
- **Permission inheritance** — All access decisions are delegated to OpenAI's existing data governance layer; the agent introduces no new permission logic.

### 9.2 Transparency & Auditability

The agent is designed for **full analytical transparency**:

1. **Reasoning exposure** — Each answer includes a summary of assumptions and execution steps.
2. **SQL linkage** — Generated queries are linked to their executed results, allowing users to inspect raw data.
3. **Step-by-step traceability** — Every intermediate step (table selection, filter application, join logic) is visible for manual verification.

This transparency model acknowledges a fundamental principle: **the agent can make mistakes**, and users must have the tools to verify every claim.

---

## 10. Lessons Learned

### Lesson 1: Tool Set Consolidation — Less Is More

**Problem:** Initial design exposed the full internal tool set to the agent, introducing **overlapping functionality** across multiple tools.

**Failure mode:** Overlapping tools that are intuitive for human manual invocation create **ambiguity for agentic systems**. When multiple tools can plausibly handle the same sub-task, the model must resolve tool selection as an additional decision variable, increasing the probability of suboptimal tool invocation:

$$
P(\text{correct\_tool} \mid q, \mathcal{T}_{\text{large}}) < P(\text{correct\_tool} \mid q, \mathcal{T}_{\text{consolidated}})
$$

where $|\mathcal{T}_{\text{consolidated}}| < |\mathcal{T}_{\text{large}}|$.

**Resolution:** Restrict and consolidate tool calls to minimize overlap, presenting the model with a **minimal, non-ambiguous** tool set. Each tool should have a clearly distinct scope with no functional overlap.

**Generalizable insight:** In agentic system design, **tool set parsimony** directly improves reliability. The agent's decision space should be as constrained as possible while remaining expressive enough for the task distribution.

### Lesson 2: Goal-Level Guidance Over Procedural Prescription

**Problem:** Highly prescriptive prompting — specifying exact step sequences — **degraded result quality**.

**Failure mode:** While many analytical questions share a general structural pattern, the details vary substantially. Rigid procedural instructions forced the model down incorrect execution paths for questions that deviated from the prescribed template.

**Resolution:** Shift to **goal-level guidance** — specifying what the agent should achieve, not how it should achieve it — and rely on GPT-5.2's reasoning capabilities to select appropriate execution paths.

**Formally:**

$$
\text{Prompt}_{\text{goal}} : \quad \text{"Determine metric } M \text{ with appropriate filters and aggregations"}
$$

outperforms:

$$
\text{Prompt}_{\text{procedural}} : \quad \text{"Step 1: Query table } T_1 \text{. Step 2: Join with } T_2 \text{ on } k \text{. Step 3: ..."}
$$

**Generalizable insight:** As model reasoning capabilities improve, **prompting should shift from procedural to declarative**. Over-specification constrains the model's ability to adapt to question-specific requirements.

### Lesson 3: Meaning Lives in Code, Not Metadata

**Problem:** Schema metadata and query history describe a table's **structure and usage patterns**, but not its **semantic meaning**.

**Failure mode:** Pipeline logic encodes critical information — filtering assumptions, freshness guarantees, business intent, temporal semantics — that **never surfaces in SQL or metadata catalogs**. Without this information, the agent cannot distinguish between tables that are structurally similar but semantically different.

**Resolution:** Crawl the production codebase with Codex to extract the **transformation logic** that produces each table, deriving a code-level semantic understanding that answers both "what's in here" and "when can I use it."

**Generalizable insight:** For any AI system operating over structured data, **the data pipeline code is a first-class source of truth** that should be systematically incorporated into the model's context. Schema catalogs are necessary but insufficient for semantic grounding.

---

## 11. Technical Analysis & Commentary

### 11.1 Architectural Significance

The data agent represents a mature instantiation of several converging architectural patterns:

| Pattern | Implementation |
|---|---|
| **Retrieval-Augmented Generation (RAG)** | Multi-source embedding-based retrieval over table metadata, code enrichments, and institutional knowledge |
| **Agentic Reasoning** | Closed-loop self-correction with multi-step goal decomposition |
| **Tool-Augmented LLMs** | SQL execution, data warehouse inspection, platform API calls |
| **Persistent Memory** | Continuously learning correction store with global/personal scoping |
| **Multi-Surface Deployment** | Unified API serving Web, Slack, IDE, and ChatGPT surfaces via MCP |

### 11.2 Context Engineering as the Primary Lever

A critical technical takeaway is that **context engineering — not model fine-tuning — is the primary lever for domain-specific agent quality**. The six-layer context hierarchy demonstrates that:

$$
\text{Agent\_Quality} = f(\text{Model\_Capability} \times \text{Context\_Richness} \times \text{Context\_Accuracy})
$$

Even GPT-5.2 produces incorrect results (e.g., vastly misestimated user counts) without appropriate context. The dramatic improvement from "22m 41s and failing" to "1m 22s and succeeding" when memory is enabled illustrates the **multiplicative relationship** between model capability and context quality.

### 11.3 Code-Level Enrichment as Differentiator

The Codex enrichment layer (Layer 3) is the **most architecturally novel** component. While schema metadata and human annotations are standard in data catalog systems, **automated extraction of semantic understanding from pipeline code** is a fundamentally different approach that:

1. **Scales without human effort** — Unlike annotations, code enrichment is generated automatically.
2. **Stays current** — Automatic regeneration tracks codebase changes.
3. **Captures implicit knowledge** — Business rules embedded in transformation logic are surfaced without requiring explicit documentation.

### 11.4 Evaluation as Continuous Quality Infrastructure

The application of the Evals API as a **continuous integration system for agent quality** is methodologically rigorous:

- **Curated golden sets** prevent evaluation gaming through memorization.
- **Semantic comparison** (not string matching) handles the many-to-one mapping from SQL syntax to semantics.
- **Dual-mode deployment** (development regression + production canary) ensures quality is protected across the full lifecycle.

### 11.5 Implications for Enterprise AI Agent Design

This system provides a reference architecture for enterprise-grade analytical agents with several transferable principles:

1. **Context layering** — Multiple complementary context sources, each addressing a distinct information gap, composed through a unified retrieval pipeline.
2. **Code as documentation** — Production codebases are the **authoritative source of data semantics** and should be systematically mined.
3. **Memory as learning** — Persistent correction stores enable continuous improvement without model retraining.
4. **Tool parsimony** — Minimal, non-overlapping tool sets improve agentic reliability.
5. **Goal-level prompting** — Declarative guidance outperforms procedural prescription for capable reasoning models.
6. **Transparent reasoning** — Agent trustworthiness requires full auditability of intermediate steps and assumptions.

---

## Summary of Key Technical Specifications

| Specification | Value |
|---|---|
| **Reasoning Model** | GPT-5.2 |
| **Code Analysis** | Codex |
| **Embedding Generation** | OpenAI Embeddings API |
| **Quality Assurance** | OpenAI Evals API |
| **Integration Protocol** | MCP (Model Context Protocol) |
| **Data Scale** | $600+$ PB, $70{,}000+$ datasets |
| **User Base** | $3{,}500+$ internal users |
| **Context Layers** | 6 (Table Usage → Human Annotations → Codex Enrichment → Institutional Knowledge → Memory → Runtime Context) |
| **Deployment Surfaces** | 4 (Web UI, Slack, IDE/CLI via MCP, Internal ChatGPT via MCP) |
| **Memory Scopes** | 2 (Global, Personal) |
| **Evaluation Mode** | Continuous (Development regression + Production canary) |
| **Security Model** | Zero-privilege-escalation pass-through |

---

*This report provides a comprehensive technical decomposition of OpenAI's in-house data agent architecture, context engineering pipeline, evaluation framework, and operational lessons. The system demonstrates that the combination of frontier model reasoning with systematic, multi-layered context engineering produces analytical agents capable of replacing multi-day manual analysis workflows with minute-scale automated insights — while maintaining the transparency and correctness guarantees required for high-stakes business decisions.*



# Inside OpenAI's In-House Data Agent: A Comprehensive Technical Report

---

## Executive Summary

This report presents an exhaustive technical analysis of OpenAI's bespoke in-house data agent — a custom-built, internal-only AI system designed to enable any employee to transition from natural-language question to data-driven insight in minutes rather than days. The agent is architected atop **GPT-5.2**, leverages **Codex** for code-level enrichment, employs the **Embeddings API** for retrieval-augmented generation (RAG), and is continuously validated via the **Evals API**. Operating across $>600$ petabytes of data, $>70{,}000$ datasets, and $>3{,}500$ internal users, the system embodies a deeply layered context architecture, closed-loop self-correction, persistent memory, and rigorous evaluation infrastructure. This report dissects every architectural layer, algorithmic decision, and systems-engineering principle underpinning the agent, providing a world-class technical reference.

---

## Table of Contents

1. [Motivation and Problem Formulation](#1-motivation-and-problem-formulation)
2. [System Architecture Overview](#2-system-architecture-overview)
3. [Entrypoints and Interface Layer](#3-entrypoints-and-interface-layer)
4. [Core Reasoning Engine: GPT-5.2](#4-core-reasoning-engine-gpt-52)
5. [The Six-Layer Context Architecture](#5-the-six-layer-context-architecture)
   - 5.1 Layer 1: Table Usage
   - 5.2 Layer 2: Human Annotations
   - 5.3 Layer 3: Codex Enrichment
   - 5.4 Layer 4: Institutional Knowledge
   - 5.5 Layer 5: Memory
   - 5.6 Layer 6: Runtime Context
6. [Retrieval-Augmented Generation Pipeline](#6-retrieval-augmented-generation-pipeline)
7. [Closed-Loop Self-Correction and Agentic Reasoning](#7-closed-loop-self-correction-and-agentic-reasoning)
8. [Conversational Multi-Turn Interaction Model](#8-conversational-multi-turn-interaction-model)
9. [Reusable Workflows](#9-reusable-workflows)
10. [Evaluation Infrastructure](#10-evaluation-infrastructure)
11. [Security, Access Control, and Transparency](#11-security-access-control-and-transparency)
12. [Lessons Learned: Engineering Principles at Scale](#12-lessons-learned-engineering-principles-at-scale)
13. [Mathematical Foundations](#13-mathematical-foundations)
14. [Future Directions](#14-future-directions)
15. [Conclusion](#15-conclusion)

---

## 1. Motivation and Problem Formulation

### 1.1 The Data Discovery Bottleneck

OpenAI's internal data platform serves:

| Dimension | Scale |
|---|---|
| Internal users | $>3{,}500$ |
| Total data volume | $>600$ PB |
| Distinct datasets | $>70{,}000$ |
| Functional teams served | Engineering, Data Science, Go-To-Market, Finance, Research |

At this scale, the **table discovery problem** becomes the dominant bottleneck. Analysts confront:

- **Schema ambiguity**: Multiple tables with overlapping column names and similar schemas but critically different semantics (e.g., one table includes logged-out users; another does not).
- **Join complexity**: Many-to-many joins, filter pushdown errors, and unhandled $\texttt{NULL}$ values can **silently invalidate** results without any runtime error.
- **Query complexity**: Production-grade analytical queries routinely exceed $180+$ lines of SQL, making correctness verification prohibitively expensive.

### 1.2 Formal Problem Statement

Let $\mathcal{D} = \{D_1, D_2, \ldots, D_N\}$ denote the set of $N = 70{,}000$ datasets. Let $q \in \mathcal{Q}$ be a natural-language question posed by a user $u$ with permission set $\mathcal{P}_u \subseteq \mathcal{D}$. The agent must solve:

$$
\hat{a} = \arg\max_{a \in \mathcal{A}} \; P\bigl(a \mid q, \; \mathcal{C}(q, \mathcal{P}_u)\bigr)
$$

where $\mathcal{C}(q, \mathcal{P}_u)$ is the **context retrieval function** that surfaces the most relevant subset of metadata, code-level enrichment, institutional knowledge, and memory — all filtered through the user's permission scope $\mathcal{P}_u$ — and $\mathcal{A}$ is the space of valid analytical answers (including SQL, visualizations, and narrative explanations).

### 1.3 Why Off-the-Shelf Solutions Fail

General-purpose text-to-SQL systems fail at OpenAI's scale for three fundamental reasons:

1. **Context poverty**: Without code-level understanding of how tables are *produced*, schema-only approaches cannot disambiguate semantically similar tables.
2. **Absence of institutional memory**: Internal terminology, metric definitions, known data incidents, and experiment-gating logic reside in Slack threads, Notion pages, and Google Docs — not in warehouse metadata.
3. **Permission-unaware generation**: External tools lack integration with OpenAI's fine-grained access-control model, creating both security and correctness risks.

---

## 2. System Architecture Overview

The agent follows a **layered agentic architecture** with clear separation of concerns:

```
┌─────────────────────────────────────────────────────────────────────┐
│                        ENTRYPOINTS                                  │
│   Agent-UI  │  Slack Agent  │  IDE Plugin  │  Codex CLI (MCP)       │
│                         │  ChatGPT Internal App (MCP Connector)     │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌──────────────────────────────────────────────────────────────────────┐
│                         AGENT-API                                    │
│  ┌────────────┐  ┌──────────────────┐  ┌─────────────────────────┐  │
│  │  Router &   │  │  Context         │  │  Tool Orchestrator      │  │
│  │  Session    │  │  Retrieval       │  │  (SQL exec, search,     │  │
│  │  Manager    │  │  Engine (RAG)    │  │   notebook publish,     │  │
│  │             │  │                  │  │   metadata service,     │  │
│  │             │  │                  │  │   Airflow, Spark)       │  │
│  └────────────┘  └──────────────────┘  └─────────────────────────┘  │
│                               │                                      │
│                               ▼                                      │
│                   ┌───────────────────────┐                          │
│                   │      GPT-5.2          │                          │
│                   │  (Reasoning Engine)   │                          │
│                   └───────────────────────┘                          │
│                               │                                      │
│              ┌────────────────┼────────────────┐                    │
│              ▼                ▼                ▼                     │
│   ┌──────────────┐  ┌──────────────┐  ┌──────────────┐             │
│   │  Data         │  │  Company      │  │  Memory      │             │
│   │  Knowledge    │  │  Context      │  │  Store       │             │
│   │  (6 layers)   │  │  (Slack,Docs) │  │              │             │
│   └──────────────┘  └──────────────┘  └──────────────┘             │
│                               │                                      │
│                               ▼                                      │
│                   ┌───────────────────────┐                          │
│                   │    Data Warehouse     │                          │
│                   │    (Live Queries)     │                          │
│                   └───────────────────────┘                          │
└──────────────────────────────────────────────────────────────────────┘
```

### 2.1 Key Architectural Properties

| Property | Implementation |
|---|---|
| **Model backbone** | GPT-5.2 (flagship reasoning model) |
| **Code enrichment** | Codex (codebase crawling and table semantics extraction) |
| **Embedding** | OpenAI Embeddings API (vector representation of enriched context) |
| **Evaluation** | OpenAI Evals API (continuous correctness measurement) |
| **Communication protocol** | Model Context Protocol (MCP) for tool interoperability |
| **Deployment surfaces** | Slack, Web UI, IDE, Codex CLI, ChatGPT Internal App |

---

## 3. Entrypoints and Interface Layer

The agent is designed to meet users **where they already work**, eliminating the friction of context-switching to a separate analytics tool.

### 3.1 Entrypoint Modalities

| Entrypoint | Protocol | Use Case |
|---|---|---|
| **Agent-UI** (Web) | HTTPS → Agent-API | Full-featured interactive analysis with visualizations |
| **Slack Agent** | Slack Events API → Agent-API | Quick ad-hoc questions in conversational context |
| **IDE Plugin** | Language Server Protocol + MCP → Agent-API | In-editor data exploration during development |
| **Codex CLI** | MCP → Agent-API | Terminal-native workflows for engineering users |
| **ChatGPT Internal App** | MCP Connector → Agent-API | Unified assistant experience with data capability |

### 3.2 Model Context Protocol (MCP)

All entrypoints communicate with the Agent-API through the **Model Context Protocol (MCP)**, which provides:

- **Standardized tool invocation**: A uniform schema for describing available tools, their parameters, and return types.
- **Bidirectional context flow**: Entrypoints can both supply context (e.g., the user's current file in an IDE) and receive structured responses.
- **Stateful sessions**: Multi-turn conversation history is maintained at the API layer, enabling seamless context carryover across turns.

---

## 4. Core Reasoning Engine: GPT-5.2

### 4.1 Role of the Reasoning Model

GPT-5.2 serves as the **central reasoning engine** — it is not merely performing text-to-SQL translation but executing a full analytical reasoning loop:

1. **Question decomposition**: Parsing ambiguous natural-language questions into structured analytical subgoals.
2. **Table selection**: Reasoning over enriched context to identify the correct tables, resolving ambiguity between semantically similar schemas.
3. **Query synthesis**: Generating syntactically and semantically correct SQL (or Spark/Python code) that accounts for join cardinality, filter semantics, and null handling.
4. **Result validation**: Inspecting intermediate results for anomalies (e.g., zero-row outputs, implausible magnitudes) and self-correcting.
5. **Narrative synthesis**: Producing human-readable explanations, including assumptions, caveats, and links to underlying data.

### 4.2 Agentic Reasoning vs. Single-Pass Generation

The critical distinction from standard text-to-SQL systems is that the agent operates in a **closed-loop agentic paradigm**:

$$
s_{t+1} = f_{\theta}\bigl(s_t, \; o_t, \; \mathcal{C}_t\bigr)
$$

where $s_t$ is the agent's internal state at step $t$, $o_t$ is the observation (e.g., query result, error message, schema inspection output), and $\mathcal{C}_t$ is the retrieved context at step $t$. The function $f_{\theta}$ (parameterized by GPT-5.2's weights $\theta$) determines the next action: issue another query, refine a filter, request clarification, or synthesize a final answer.

This stands in contrast to **single-pass generation** where a model produces one SQL query and returns the result without validation or iteration.

---

## 5. The Six-Layer Context Architecture

The agent's accuracy is fundamentally determined by the quality and depth of the context it can access. The architecture implements six distinct, complementary context layers, each addressing a specific category of information that is necessary for correct data analysis.

### 5.1 Layer 1: Table Usage

**Purpose**: Ground the agent in empirical patterns of how tables are actually used.

**Components**:

| Signal | Description | Value to Agent |
|---|---|---|
| **Schema metadata** | Column names, data types, partitioning keys | Informs syntactically correct SQL generation |
| **Table lineage** | Upstream sources, downstream consumers, DAG position | Reveals relationships between tables and data flow |
| **Historical query logs** | Past SQL queries executed against each table | Provides empirical templates for correct join patterns and filters |

**Technical Implementation**:

- Schema metadata is extracted from the data warehouse's information schema.
- Lineage is derived from Airflow DAG definitions and Spark job configurations.
- Query logs are parsed and deduplicated, with frequency-weighted indexing to prioritize commonly used patterns.

**Formal Representation**: For a table $T_i$, its usage context is:

$$
\mathcal{U}(T_i) = \bigl\langle \text{Schema}(T_i), \; \text{Lineage}(T_i), \; \text{QueryHistory}(T_i) \bigr\rangle
$$

### 5.2 Layer 2: Human Annotations

**Purpose**: Capture expert-authored semantic descriptions that encode business intent, known caveats, and domain-specific meaning that cannot be inferred from schemas or query logs.

**Characteristics**:

- Written by domain experts (data engineers, analysts, product managers).
- Describe **why** a table exists, **what business question** it answers, and **known limitations** (e.g., "This table excludes logged-out users," "Revenue figures are pre-tax").
- Cover both table-level and column-level annotations.

**Why This Layer Is Necessary**: Metadata alone cannot disambiguate between two tables with identical schemas but different semantics. Human annotations provide the **intentional** dimension — the "why" behind the "what."

### 5.3 Layer 3: Codex Enrichment

**Purpose**: Derive a **code-level definition** of each table by crawling the codebase that produces it, extracting semantics that are invisible in SQL or metadata.

This is arguably the most technically novel layer of the architecture. The key insight is:

> **"A table's true meaning lives in the code that produces it."**

**Codex Enrichment Pipeline**:

```
Popular / High-Usage Tables
        │
        ▼
┌───────────────────────────────┐
│     Codex Task Orchestrator   │
│                               │
│  ┌─────────────────────────┐  │
│  │ Task 1: Table Purpose   │  │ ← Why does this table exist?
│  │ Task 2: Grain & Keys    │  │ ← What is the primary key? What is each row?
│  │ Task 3: Downstream Use  │  │ ← How is this table consumed?
│  │ Task 4: Alternatives    │  │ ← What similar tables exist and how do they differ?
│  │ Task 5: Data Freshness  │  │ ← How often is this table updated?
│  └─────────────────────────┘  │
│                               │
│  Source: OpenAI Codebase      │
│  (Pipeline definitions,       │
│   Spark jobs, ETL logic,      │
│   experiment configurations)  │
└───────────────────────────────┘
        │
        ▼
  Enriched Table Profile
```

**Information Extracted by Codex**:

| Enrichment Dimension | Description | Example |
|---|---|---|
| **Table purpose** | Natural-language summary of what the table represents | "Daily aggregation of first-party ChatGPT web traffic, excluding API-sourced events" |
| **Grain and primary keys** | The level of granularity of each row | "One row per user per day per product surface" |
| **Value uniqueness** | Whether columns contain unique values or are many-to-one | "user_id is unique per row; session_id is not" |
| **Update frequency** | How often the table is refreshed | "Updated daily at 04:00 UTC via Airflow DAG `chatgpt_daily_agg`" |
| **Scope and exclusions** | What data is explicitly included or excluded | "Excludes internal test accounts and logged-out sessions" |
| **Downstream usage patterns** | How the table is consumed in Spark, Python, and other systems beyond SQL | "Joined with `dim_user_segments` in the weekly retention pipeline" |
| **Alternative tables** | Similar tables and their distinguishing features | "Similar to `chatgpt_traffic_v2` but includes API traffic" |

**Automatic Refresh**: Codex enrichment is regenerated automatically, ensuring that as pipeline code changes, the agent's understanding of each table stays current without manual maintenance.

**Formal Representation**: For a table $T_i$:

$$
\mathcal{E}(T_i) = \text{Codex}\bigl(\text{Codebase}, \; T_i\bigr) = \bigl\langle \text{Purpose}, \; \text{Grain}, \; \text{Keys}, \; \text{Freshness}, \; \text{Scope}, \; \text{Alternatives} \bigr\rangle
$$

### 5.4 Layer 4: Institutional Knowledge

**Purpose**: Provide access to the **organizational context** that surrounds data — product launches, reliability incidents, internal codenames, canonical metric definitions, and experiment configurations.

**Sources**:

| Source | Content Type | Example |
|---|---|---|
| **Slack** | Real-time discussions, incident postmortems | "The usage dip in December was caused by a logging issue starting Nov 13, 2025" |
| **Google Docs** | Metric definitions, launch plans, design documents | "WAU is defined as distinct users with $\geq 1$ qualifying action in a 7-day window" |
| **Notion** | Team wikis, process documentation, internal tool guides | "The experiment gate `img_gen_v2` uses string matching against `experiment_config.gate_name`" |

**Ingestion Pipeline**:

1. Documents are crawled from each source with their associated metadata (author, team, creation date, access permissions).
2. Content is chunked, embedded using the **OpenAI Embeddings API**, and stored in a vector database.
3. At retrieval time, a **permission-aware retrieval service** ensures that the agent only surfaces documents the querying user is authorized to access.
4. A caching layer minimizes redundant embedding computations and API calls.

**Why This Layer Is Critical**: Without institutional knowledge, the agent cannot explain anomalies. For example, a sudden dip in connector usage in December 2025 would be unexplainable from data alone — the true cause was a logging infrastructure change following the ChatGPT 5.1 launch, documented only in Slack postmortems.

### 5.5 Layer 5: Memory

**Purpose**: Enable **continuous self-improvement** by persisting corrections, discovered nuances, and non-obvious analytical constraints across sessions.

**Memory Taxonomy**:

| Memory Scope | Visibility | Use Case |
|---|---|---|
| **Global memory** | All users | Corrections that apply universally (e.g., "Table X excludes logged-out users") |
| **Personal memory** | Individual user | User-specific preferences and context (e.g., "When I ask about 'growth,' I mean WoW percentage change") |

**Memory Lifecycle**:

1. **Trigger**: The user provides a correction, or the agent discovers a non-obvious constraint during analysis.
2. **Proposal**: The agent surfaces a notification: *"Data agent wants to save 2 learnings to memory."*
3. **User confirmation**: The user reviews and approves (or edits) the proposed memory.
4. **Persistence**: The memory is stored with metadata (scope, timestamp, source conversation).
5. **Retrieval**: On subsequent queries, relevant memories are retrieved and injected into the agent's context.
6. **Manual management**: Users can create, edit, and delete memories through a dedicated interface.

**Why Memory Is Necessary**: Some analytical constraints are **impossible to infer** from schema, code, or documentation alone. The canonical example from the report: filtering for a particular analytics experiment required matching against a specific string defined in an experiment gate. Without memory of this correction, the agent would repeatedly attempt fuzzy string matching and produce incorrect results.

**Formal Representation**: The memory store $\mathcal{M}$ is a set of tuples:

$$
\mathcal{M} = \bigl\{ (m_j, \; \text{scope}_j, \; \text{timestamp}_j, \; \text{source}_j) \bigr\}_{j=1}^{|\mathcal{M}|}
$$

where each $m_j$ is a structured learning (e.g., a filter condition, a table preference, a metric definition correction).

### 5.6 Layer 6: Runtime Context

**Purpose**: Provide **live, real-time** information when no prior context exists or when existing information may be stale.

**Capabilities**:

| Capability | Mechanism |
|---|---|
| **Live schema inspection** | Direct queries to the data warehouse's information schema |
| **Data sampling** | Executing `SELECT` queries on live data to validate assumptions |
| **Platform system queries** | Communicating with the metadata service, Airflow (DAG status, run history), and Spark (job status, cluster health) |

**When Runtime Context Is Invoked**:

- A table has never been queried before (no historical query logs, no Codex enrichment).
- Existing enrichment may be outdated (e.g., a schema change occurred after the last Codex crawl).
- The agent needs to verify an intermediate result against live data.

---

## 6. Retrieval-Augmented Generation Pipeline

### 6.1 Offline Preprocessing

A **daily offline pipeline** aggregates and normalizes context from Layers 1–5 into a unified representation:

```
┌──────────────────────────────────────────────────────────┐
│                  DAILY OFFLINE PIPELINE                   │
│                                                          │
│  Table Usage ──┐                                         │
│  Human         │                                         │
│  Annotations ──┼──▶ Normalize & Merge ──▶ Enriched      │
│  Codex         │                          Table          │
│  Enrichment ──┘                          Profiles        │
│  Institutional                              │            │
│  Knowledge ────────────────────────────────▶│            │
│  Memory ───────────────────────────────────▶│            │
│                                              │            │
│                                              ▼            │
│                                    OpenAI Embeddings API  │
│                                              │            │
│                                              ▼            │
│                                    Vector Store           │
│                                    (Embeddings + Metadata)│
└──────────────────────────────────────────────────────────┘
```

### 6.2 Embedding Generation

Each enriched table profile $\mathcal{P}(T_i) = \bigl\langle \mathcal{U}(T_i), \; \text{Annotations}(T_i), \; \mathcal{E}(T_i) \bigr\rangle$ is serialized into text and passed through the **OpenAI Embeddings API** to produce a dense vector representation:

$$
\mathbf{e}_i = \text{Embed}\bigl(\text{Serialize}(\mathcal{P}(T_i))\bigr) \in \mathbb{R}^d
$$

where $d$ is the embedding dimensionality.

### 6.3 Query-Time Retrieval

At query time, the user's natural-language question $q$ is embedded:

$$
\mathbf{e}_q = \text{Embed}(q) \in \mathbb{R}^d
$$

The $k$ most relevant table profiles are retrieved via approximate nearest-neighbor search:

$$
\text{TopK}(q) = \underset{T_i \in \mathcal{D}}{\arg\text{top-}k} \; \text{sim}(\mathbf{e}_q, \; \mathbf{e}_i)
$$

where $\text{sim}(\cdot, \cdot)$ is cosine similarity:

$$
\text{sim}(\mathbf{e}_q, \; \mathbf{e}_i) = \frac{\mathbf{e}_q \cdot \mathbf{e}_i}{\|\mathbf{e}_q\| \; \|\mathbf{e}_i\|}
$$

### 6.4 Hybrid Retrieval

The system employs **hybrid retrieval** combining:

- **Semantic search**: Embedding-based retrieval as described above, effective for capturing conceptual similarity.
- **Exact text retrieval**: Keyword and metadata-based matching for precise identifiers (table names, column names, internal codenames).

This dual approach ensures that both **conceptually similar** and **lexically exact** matches are surfaced.

### 6.5 Scalability Properties

| Property | Value |
|---|---|
| Number of indexed tables | $>70{,}000$ |
| Daily refresh cycle | Full reindexing of enriched profiles |
| Runtime latency | Predictable and low (sub-second retrieval) |
| Incremental updates | Memory and new tables are indexed as they are created |

---

## 7. Closed-Loop Self-Correction and Agentic Reasoning

### 7.1 The Agentic Loop

The agent does **not** follow a fixed script. Instead, it operates as a **stateful reasoning agent** that evaluates its own progress and self-corrects:

```
┌─────────────────────────────────────────────────┐
│              AGENTIC REASONING LOOP              │
│                                                  │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐   │
│  │ Formulate│───▶│ Execute  │───▶│ Validate │   │
│  │ Plan     │    │ Action   │    │ Result   │   │
│  └──────────┘    └──────────┘    └─────┬────┘   │
│       ▲                                │        │
│       │              ┌─────────────────┘        │
│       │              ▼                          │
│       │      ┌──────────────┐                   │
│       │      │ Result OK?   │                   │
│       │      │              │                   │
│       │      │  Yes ──▶ Synthesize Answer       │
│       │      │  No  ──▶ Diagnose & Revise       │
│       │      └──────┬───────┘                   │
│       │             │ (No)                      │
│       └─────────────┘                           │
└─────────────────────────────────────────────────┘
```

### 7.2 Self-Correction Mechanisms

| Failure Mode | Detection | Correction |
|---|---|---|
| Zero-row result from incorrect join | Row count check on intermediate CTEs | Inspect join keys, adjust join type or filter conditions |
| Implausible magnitude | Sanity check against known benchmarks or historical ranges | Investigate aggregation level, check for duplication |
| SQL execution error | Database error message parsing | Analyze error, fix syntax or schema reference, retry |
| Stale data | Compare data freshness against query time range | Fall back to alternative table or issue warning |

### 7.3 Context Carryover

Critically, the agent **retains full context** across self-correction iterations. Each observation $o_t$ and the reasoning that led to revision are preserved in the agent's working memory, ensuring that:

- The same mistake is not repeated within a session.
- Learnings from early steps inform later steps.
- The user can inspect the full reasoning trace for transparency.

---

## 8. Conversational Multi-Turn Interaction Model

### 8.1 Design Philosophy

The agent is designed to behave like **a teammate, not a tool**. This distinction manifests in several key behaviors:

| Behavior | Description |
|---|---|
| **Context carryover** | Full conversation history is maintained across turns; users never need to restate prior context |
| **Mid-analysis interruption** | Users can redirect the agent during an ongoing analysis without losing progress |
| **Proactive clarification** | When instructions are ambiguous, the agent asks clarifying questions before proceeding |
| **Sensible defaults** | When clarification is not provided, the agent applies reasonable defaults (e.g., assuming the last 30 days for a time-unspecified query) and states its assumptions explicitly |
| **Bidirectional exploration** | Equally effective for precise queries ("Tell me about this table") and open-ended exploration ("I'm seeing a dip here — can we break this down by customer type and timeframe?") |

### 8.2 Interaction Model Formalization

The multi-turn interaction can be modeled as a partially observable Markov decision process (POMDP):

$$
\bigl\langle \mathcal{S}, \; \mathcal{A}, \; \mathcal{O}, \; T, \; \Omega, \; R \bigr\rangle
$$

where:

- $\mathcal{S}$: State space (user intent, data state, conversation history)
- $\mathcal{A}$: Action space (generate SQL, ask clarification, issue tool call, synthesize answer)
- $\mathcal{O}$: Observation space (query results, error messages, user responses)
- $T: \mathcal{S} \times \mathcal{A} \rightarrow \mathcal{S}$: State transition function
- $\Omega: \mathcal{S} \times \mathcal{A} \rightarrow \mathcal{O}$: Observation function
- $R: \mathcal{S} \times \mathcal{A} \rightarrow \mathbb{R}$: Reward function (user satisfaction, correctness, latency)

The agent's policy $\pi_\theta(a \mid o_{1:t}, q)$ is implicitly defined by GPT-5.2's reasoning capabilities, conditioned on the full observation history and the original question.

---

## 9. Reusable Workflows

### 9.1 Motivation

Post-deployment observation revealed that users frequently executed **identical or structurally similar analyses** for routine tasks:

- Weekly business health reports
- Table validation checks
- Launch impact assessments
- Metric trend monitoring

### 9.2 Workflow Architecture

Workflows are **reusable instruction sets** that encode:

| Component | Description |
|---|---|
| **Analysis template** | The sequence of analytical steps (e.g., pull metric, compute WoW change, segment by geography) |
| **Context bindings** | Which tables, metrics, and filters to use |
| **Best practices** | Known pitfalls, correct join patterns, and validated transformations |
| **Output format** | Structured output (table, chart, narrative) matching the user's expectation |

### 9.3 Benefits

- **Consistency**: The same analysis produces the same result regardless of which user runs it.
- **Efficiency**: Eliminates repeated discovery and reasoning for known analytical patterns.
- **Knowledge encoding**: Best practices are captured once and reused across the organization.

Users access workflows through a dedicated UI element ("Use a workflow") that presents available templates relevant to their role and permissions.

---

## 10. Evaluation Infrastructure

### 10.1 Evaluation Philosophy

> "Without a tight feedback loop, regressions are inevitable and invisible."

The agent's evaluation infrastructure is designed to provide **continuous, automated correctness measurement** that catches regressions before they reach users.

### 10.2 Evaluation Pipeline Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    EVALUATION PIPELINE                        │
│                                                              │
│  Curated Q&A Eval Pairs                                      │
│  ┌─────────────────────────────────────────┐                 │
│  │  Question: "What was ChatGPT DAU        │                 │
│  │            on 2025-10-06?"              │                 │
│  │  Expected SQL: SELECT ... FROM ...      │                 │
│  │  Expected Result: {dau: 800_000_000}    │                 │
│  └─────────────┬───────────────────────────┘                 │
│                │                                             │
│       ┌────────┴────────┐                                    │
│       ▼                 ▼                                    │
│  ┌──────────┐    ┌──────────────┐                            │
│  │ Agent    │    │ Execute      │                            │
│  │ Generates│    │ Expected     │                            │
│  │ SQL      │    │ SQL          │                            │
│  └────┬─────┘    └──────┬───────┘                            │
│       │                 │                                    │
│       ▼                 ▼                                    │
│  ┌──────────┐    ┌──────────────┐                            │
│  │ Execute  │    │ Expected     │                            │
│  │ Generated│    │ Result Set   │                            │
│  │ SQL      │    │              │                            │
│  └────┬─────┘    └──────┬───────┘                            │
│       │                 │                                    │
│       └────────┬────────┘                                    │
│                ▼                                             │
│       ┌──────────────────┐                                   │
│       │  OpenAI Evals    │                                   │
│       │  Grader          │                                   │
│       │                  │                                   │
│       │  Inputs:         │                                   │
│       │  - Generated SQL │                                   │
│       │  - Generated     │                                   │
│       │    Result Set    │                                   │
│       │  - Expected SQL  │                                   │
│       │  - Expected      │                                   │
│       │    Result Set    │                                   │
│       │                  │                                   │
│       │  Outputs:        │                                   │
│       │  - Score ∈ [0,1] │                                   │
│       │  - Reasoning     │                                   │
│       └──────────────────┘                                   │
└──────────────────────────────────────────────────────────────┘
```

### 10.3 Evaluation Methodology

The evaluation is **not** based on naive string matching of SQL. This is critical because:

1. **Syntactic variation**: Two SQL queries can differ syntactically while being semantically identical:
   ```sql
   -- Generated
   SELECT COUNT(DISTINCT user_id) AS dau FROM events WHERE dt = '2025-10-06'
   
   -- Expected
   SELECT COUNT(DISTINCT user_id) dau FROM events WHERE dt = '2025-10-06'
   ```
   These are equivalent but textually different.

2. **Acceptable result variation**: The generated query may include additional columns that do not materially affect the answer.

3. **Semantic equivalence under rewriting**: Different join orders, subquery vs. CTE formulations, and filter placements can produce identical results.

**Dual Comparison Strategy**:

| Comparison Type | Method | Purpose |
|---|---|---|
| **SQL comparison** | Structural and semantic analysis of query logic | Detect logically equivalent but syntactically different queries |
| **Dataframe comparison** | Row-by-row and column-by-column result comparison | Verify that the actual output data matches expected data |

The OpenAI Evals grader consumes both signals and produces:

- **Score** $\in [0, 1]$: Quantifying correctness.
- **Reasoning**: A natural-language explanation of the grading decision, capturing both correctness and acceptable variation.

### 10.4 Evaluation Deployment Model

| Deployment Phase | Purpose |
|---|---|
| **During development** | Unit-test-like regression detection for every code change |
| **Canary in production** | Continuous monitoring of live agent behavior against known-correct baselines |
| **Post-incident** | Root-cause analysis when user-reported errors occur |

### 10.5 Formal Evaluation Metric

For a set of $n$ evaluation pairs $\{(q_i, \hat{s}_i)\}_{i=1}^n$, where $q_i$ is the question and $\hat{s}_i$ is the expected SQL, the agent's overall correctness score is:

$$
\text{Score}_{\text{overall}} = \frac{1}{n} \sum_{i=1}^{n} \text{EvalGrader}\bigl(\text{SQL}_{\text{gen}}(q_i), \; \text{Result}_{\text{gen}}(q_i), \; \hat{s}_i, \; \text{Result}_{\text{expected}}(\hat{s}_i)\bigr)
$$

---

## 11. Security, Access Control, and Transparency

### 11.1 Security Model: Pass-Through Access Control

The agent operates as a **pure interface layer** — it inherits and enforces the existing security and access-control model of OpenAI's data platform. It introduces **no new privilege escalation vectors**.

| Principle | Implementation |
|---|---|
| **Pass-through permissions** | Users can only query tables they already have permission to access via the underlying data platform |
| **Permission-aware retrieval** | The RAG retrieval service filters context based on user permissions, ensuring that even table descriptions and enrichments are only surfaced for authorized tables |
| **Graceful degradation** | When access is missing, the agent either flags the issue or falls back to alternative datasets the user is authorized to use |
| **No data caching bypass** | The agent does not cache query results in a way that could bypass access controls |

### 11.2 Transparency and Auditability

| Feature | Benefit |
|---|---|
| **Reasoning summary** | Each answer includes a summary of assumptions and execution steps |
| **Direct result links** | Generated SQL and query results are linked, allowing users to inspect raw data |
| **Step-by-step trace** | The full reasoning trace (including self-corrections) is visible to the user |
| **Explicit assumption disclosure** | When the agent applies defaults (e.g., time range), these are stated explicitly |

### 11.3 Trust Calibration

The system is designed with the explicit acknowledgment that **it can make mistakes**. Transparency mechanisms serve as **trust calibration devices**, enabling users to:

1. Verify the agent's reasoning against their domain expertise.
2. Identify incorrect assumptions early in the analysis.
3. Provide corrections that are persisted in memory for future improvement.

---

## 12. Lessons Learned: Engineering Principles at Scale

### 12.1 Lesson 1: Less Is More — Tool Consolidation

**Observation**: Early versions exposed the full tool set to the agent, resulting in **overlapping functionality** that created ambiguity during tool selection.

**Problem Analysis**: When multiple tools can accomplish the same task (e.g., two different metadata lookup endpoints), the agent must expend reasoning capacity on tool selection rather than problem-solving. This redundancy, while manageable for human users who can read documentation, creates a combinatorial decision burden for the agent.

**Resolution**: Restrict and consolidate tool calls, ensuring that each tool has a **unique, non-overlapping purpose**. The principle can be formalized:

$$
\forall \; t_i, t_j \in \mathcal{T}, \quad i \neq j \implies \text{Capability}(t_i) \cap \text{Capability}(t_j) = \emptyset
$$

where $\mathcal{T}$ is the set of tools available to the agent.

### 12.2 Lesson 2: Guide the Goal, Not the Path

**Observation**: Highly prescriptive prompting — specifying exact step sequences — **degraded performance**.

**Problem Analysis**: While many analytical questions share a general shape, the specific details vary enough that rigid instructions often pushed the agent down incorrect execution paths. Overly prescriptive prompts reduce the agent's ability to leverage its reasoning capabilities to adapt to novel situations.

**Resolution**: Shift to **high-level goal-oriented prompting**, specifying *what* the agent should achieve (e.g., "identify the correct table for this metric") rather than *how* (e.g., "first query the metadata service, then inspect the schema, then..."). This relies on GPT-5.2's reasoning to determine the optimal execution path for each specific question.

**Principle**:

$$
\text{Quality}(\text{Agent Output}) \propto \text{Goal Clarity} \times \text{Reasoning Freedom}
$$

### 12.3 Lesson 3: Meaning Lives in Code

**Observation**: Schemas and query history describe a table's **shape and usage**, but its **true semantics** — assumptions, freshness guarantees, business intent, exclusion criteria — reside in the **pipeline code** that produces it.

**Problem Analysis**: Two tables may have identical schemas and similar names but represent fundamentally different data populations (e.g., one includes API traffic, the other does not). This distinction is encoded only in the ETL logic that populates each table.

**Resolution**: Use **Codex** to crawl the codebase, extracting code-level definitions for each table. This enables the agent to answer not just "what's in here" but "when can I use it" and "how is it different from that similar table" with far greater accuracy than warehouse metadata alone.

---

## 13. Mathematical Foundations

### 13.1 Information-Theoretic Framing of Context Layers

Each context layer reduces the **entropy** of the agent's posterior distribution over possible answers. Let $H(A \mid Q)$ be the entropy of the answer distribution given only the question:

$$
H(A \mid Q) = -\sum_{a \in \mathcal{A}} P(a \mid Q) \log P(a \mid Q)
$$

Each additional context layer $\mathcal{C}_\ell$ provides **mutual information** that reduces this entropy:

$$
I(A; \mathcal{C}_\ell \mid Q) = H(A \mid Q) - H(A \mid Q, \mathcal{C}_\ell) \geq 0
$$

The cumulative effect of all six layers is:

$$
H(A \mid Q, \mathcal{C}_1, \mathcal{C}_2, \ldots, \mathcal{C}_6) \ll H(A \mid Q)
$$

This formalizes the intuition that **more contextual grounding leads to more deterministic (and thus more accurate) agent behavior**.

### 13.2 Retrieval Relevance Optimization

The RAG pipeline must maximize retrieval precision while maintaining recall. For a query $q$ and a relevance threshold $\tau$, the retrieved set is:

$$
\mathcal{R}(q, \tau) = \bigl\{ T_i \in \mathcal{D} \;\big|\; \text{sim}(\mathbf{e}_q, \mathbf{e}_i) \geq \tau \bigr\}
$$

The objective is to select $\tau$ such that:

$$
\tau^* = \arg\max_{\tau} \; F_\beta\bigl(\text{Precision}(\mathcal{R}(q, \tau)), \; \text{Recall}(\mathcal{R}(q, \tau))\bigr)
$$

where $F_\beta$ is the $F$-score with parameter $\beta$ controlling the precision-recall tradeoff, tuned based on the cost asymmetry between missing a relevant table (false negative) and including an irrelevant one (false positive).

### 13.3 Self-Correction as Iterative Bayesian Updating

Each self-correction step can be interpreted as a Bayesian update. Let $\theta$ represent the agent's belief about the correct analytical approach. After observing result $o_t$:

$$
P(\theta \mid o_{1:t}) = \frac{P(o_t \mid \theta) \cdot P(\theta \mid o_{1:t-1})}{P(o_t \mid o_{1:t-1})}
$$

When $o_t$ is anomalous (e.g., zero rows), $P(o_t \mid \theta)$ is low for the current approach $\theta$, causing a significant posterior update toward alternative approaches.

### 13.4 Memory as Persistent Prior

The memory system $\mathcal{M}$ functions as a **persistent prior** that modifies the agent's initial distribution:

$$
P(a \mid q, \mathcal{C}) \rightarrow P(a \mid q, \mathcal{C}, \mathcal{M})
$$

Memories inject high-confidence constraints (e.g., "this experiment uses this specific filter string") that sharply narrow the posterior, preventing the agent from exploring incorrect regions of the solution space.

---

## 14. Future Directions

The report identifies three key axes of continued development:

### 14.1 Handling Ambiguity

Improving the agent's ability to handle **inherently ambiguous questions** — cases where multiple valid interpretations exist and the correct one depends on unstated user intent or organizational context not yet captured in any layer.

### 14.2 Reliability and Validation

Strengthening the agent's **self-validation capabilities** — developing more sophisticated checks for result plausibility, including automated comparison against historical baselines and cross-validation across alternative analytical approaches.

### 14.3 Deeper Workflow Integration

Moving from a **separate tool** paradigm to a **seamlessly embedded** paradigm — the agent should blend naturally into existing work surfaces (dashboards, notebooks, CI/CD pipelines) rather than requiring users to context-switch to interact with it.

### 14.4 Continuous Improvement Through Underlying Model Advances

As GPT-5.2 and successor models improve in reasoning, validation, and self-correction, the agent will inherit these improvements directly. The architecture is designed such that **model improvements translate directly into agent capability gains** without architectural rearchitecture.

---

## 15. Conclusion

OpenAI's in-house data agent represents a mature implementation of **agentic AI for enterprise data analysis** at scale. Its architecture embodies several key technical innovations:

1. **Six-layer context architecture**: A principled hierarchy of context sources — from schema metadata through code-level semantics to institutional knowledge and persistent memory — that systematically reduces uncertainty and grounds the agent in the organization's data reality.

2. **Codex-powered code-level enrichment**: The insight that a table's true meaning lives in the code that produces it, operationalized through automated codebase crawling that extracts semantics invisible to metadata-only approaches.

3. **Closed-loop agentic reasoning**: A self-correcting execution paradigm where the agent evaluates its own intermediate results, diagnoses failures, and revises its approach — shifting iteration burden from the user to the system.

4. **Persistent memory system**: A continuously learning mechanism that accumulates non-obvious corrections and constraints, ensuring that the agent's baseline accuracy improves with every interaction.

5. **Rigorous evaluation infrastructure**: Continuous correctness measurement via the OpenAI Evals API, treating evaluation as a first-class engineering concern rather than an afterthought.

6. **Pass-through security model**: A zero-privilege-escalation design that inherits and enforces existing access controls, maintaining trust at enterprise scale.

The system transforms data analysis from a **multi-day, specialist-dependent process** into a **minutes-long, universally accessible interaction** — while maintaining the correctness, nuance, and institutional awareness that enterprise-grade decisions demand.

---

*Built with GPT-5.2 · Codex · OpenAI Embeddings API · OpenAI Evals API · Model Context Protocol (MCP)*